# SuitCVLO en Google Colab — lote de 40 fotos

Procesa fotos de publicidad exterior con **la misma secuencia** que la app local
(`SuitCVLO-open`): GPS EXIF → cruce con PINs por radio → estructura
`<AAMMPQ>\auditoria\<Grupo>\<PIN>\` + `info.txt` + `idxfotos.xlsx`.

**Lo que cambia aquí**: la IA corre **dentro de Colab con la GPU gratis** (sin keys, sin créditos).

## Cómo se usa (4 pasos)

1. **Entorno de ejecución → Cambiar tipo de entorno → GPU** (T4 o la que te toque).
2. Ejecuta las celdas **en orden** (▶).
3. En la celda *Parámetros* revisa las rutas de tu Google Drive.
4. La celda *Lote* toma **las 40 fotos pendientes** (las que todavía no están en `idxfotos.xlsx`
   ni copiadas en la salida) y las procesa. **Nunca repite.**

> Requisito: haber copiado la carpeta `SuitCVLOGCPy` a tu Google Drive (ver `INSTRUCCIONES.md`).


In [ ]:
#@title 1) Entorno: GPU + dependencias
import subprocess, sys

def sh(cmd):
    print('$ ' + cmd)
    subprocess.run(cmd, shell=True)

# --- GPU obligatoria ---
try:
    import torch
    if torch.cuda.is_available():
        libre = torch.cuda.mem_get_info()[0] / 1024 ** 3
        print('GPU:', torch.cuda.get_device_name(0), '| VRAM libre: {:.1f} GB'.format(libre))
    else:
        print('!! SIN GPU — menú: Entorno de ejecución ▸ Cambiar tipo de entorno ▸ GPU')
except Exception as e:
    print('torch no encontrado:', e)

# --- dependencias ---
sh(f'{sys.executable} -m pip install -q -U pillow-heif openpyxl gdown pillow accelerate qwen-vl-utils')
sh(f'{sys.executable} -m pip install -q "transformers>=4.51.0"')

import PIL, openpyxl, pillow_heif
print('Pillow', PIL.__version__, '| openpyxl', openpyxl.__version__, '→ OK')

In [ ]:
#@title 2) Google Drive + rutas del proyecto
from google.colab import drive
drive.mount('/content/drive')

BASE          = '/content/drive/MyDrive/cvlo'                       #@param {type:"string"}
CARPETA_FOTOS = '/content/drive/MyDrive/cvlo/Recorrido'             #@param {type:"string"}
SALIDA        = '/content/drive/MyDrive/cvlo/salida'                #@param {type:"string"}
RUTA_PINES    = '/content/drive/MyDrive/cvlo/GooMapsPines.txt'      #@param {type:"string"}
RUTA_KML      = '/content/drive/MyDrive/cvlo/RutaKML.kml'           #@param {type:"string"}
RUTA_IDX      = '/content/drive/MyDrive/cvlo/idxfotos.xlsx'         #@param {type:"string"}
TRABAJO       = '/content/drive/MyDrive/cvlo/trabajo_colab'         #@param {type:"string"}

N_FOTOS = 40   #@param {type:"slider", min:1, max:200, step:1}
RADIO   = 250  #@param {type:"slider", min:50, max:1000, step:50}
CAP     = 10   #@param {type:"slider", min:1, max:50, step:1}

import os
faltan = [p for p in (CARPETA_FOTOS, RUTA_PINES, RUTA_KML) if not os.path.exists(p)]
if faltan:
    print('RUTAS QUE NO EXISTEN EN DRIVE:')
    for p in faltan:
        print('  ✗', p)
    raise SystemExit('Corrige la celda 2 y vuelve a ejecutarla.')

# El xlsx: si ya existe en Drive se usa tal cual (mismo índice que la app local);
# si no, esta corrida lo crea con las 10 columnas.
print('idxfotos:', RUTA_IDX, '→', 'existe' if os.path.exists(RUTA_IDX)
      and os.path.getsize(RUTA_IDX) > 0 else 'SE CREARÁ')
print('Rutas OK ·', N_FOTOS, 'fotos · radio', RADIO, 'm · tope', CAP)

In [ ]:
#@title 3) Importar el pipeline (la MISMA lógica que la app local)
PROYECTO = '/content/drive/MyDrive/SuitCVLOGCPy'                  #@param {type:"string"}

import sys, os
if not os.path.exists(os.path.join(PROYECTO, 'proceso.py')):
    raise SystemExit(
        'No encuentro proceso.py en {}\n'
        'Copia la carpeta SuitCVLOGCPy a tu Google Drive (ver INSTRUCCIONES.md) '
        'y corrige PROYECTO.'.format(PROYECTO))

sys.path.insert(0, PROYECTO)
import motor, proceso, utilidades as U
from ia import switch

print('pipeline importado de', PROYECTO)
print('cadena:', ' → '.join(switch.ORDEN))
print('encabezado idxfotos:', motor.IDX_HEADERS)

In [ ]:
#@title 4) Cargar el modelo de visión en la GPU (primer eslabón)
import os, time, torch
from transformers import AutoProcessor

try:
    from transformers import AutoModelForImageTextToText as Modelo
except ImportError:                     # transformers viejo
    from transformers import AutoModelForVision2Seq as Modelo

libre = torch.cuda.mem_get_info()[0] / 1024 ** 3 if torch.cuda.is_available() else 0
MODELO_FUERZA = ''   #@param {type:"string"}
# en blanco = automático: 7B si hay 15 GB+ de VRAM, si no el 3B (cabe en T4)
NOMBRE = MODELO_FUERZA or ('Qwen/Qwen2.5-VL-7B-Instruct' if libre >= 15
                           else 'Qwen/Qwen2.5-VL-3B-Instruct')
print('VRAM libre: {:.1f} GB → modelo {}'.format(libre, NOMBRE))

t0 = time.time()
PROC = AutoProcessor.from_pretrained(NOMBRE)
MODEL = Modelo.from_pretrained(NOMBRE, torch_dtype=torch.float16, device_map='auto')
MODEL.eval()
print('modelo cargado en {:.0f} s'.format(time.time() - t0))

def leer_vlm(ruta, prompt):
    """Firma que el switch espera: (ruta, prompt) -> texto."""
    from qwen_vl_utils import process_vision_info
    mensajes = [{'role': 'user', 'content': [
        {'type': 'image', 'image': ruta, 'max_pixels': 1024 * 1024},
        {'type': 'text', 'text': prompt}]}]
    texto = PROC.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    im, vid = process_vision_info(mensajes)
    inp = PROC(text=[texto], images=im, videos=vid, padding=True,
               return_tensors='pt').to(MODEL.device)
    with torch.no_grad():
        out = MODEL.generate(**inp, max_new_tokens=600, do_sample=False)
    out = out[:, inp.input_ids.shape[1]:]
    return PROC.batch_decode(out, skip_special_tokens=True)[0].strip()

switch.registrar_vlm(leer_vlm, os.path.basename(NOMBRE) + ' (GPU Colab)')
print('registrado como primer eslabón →', switch._disponible('colab'))

In [ ]:
#@title 5) Smoke: leer 1 foto ANTES del lote
import glob, time

candidatas = sorted(p for p in glob.glob(os.path.join(CARPETA_FOTOS, '*')) if U.es_imagen(p))
if not candidatas:
    raise SystemExit('No hay imágenes en ' + CARPETA_FOTOS)
prueba = candidatas[0]
print('foto de prueba:', os.path.basename(prueba))

t0 = time.time()
texto = switch.leer(prueba, motor.PROMPT_LECTURA, proveedor='auto')
print('→ {:.1f} s'.format(time.time() - t0))
print('-' * 60)
print(texto)
print('-' * 60)
print('Si esto salió bien, ejecuta la celda siguiente.')

In [ ]:
#@title 6) Lote: las 40 fotos pendientes
switch.reset()          # sin cooldowns heredados de una corrida anterior
TOTAL = 0

def progreso(c):
    global TOTAL
    if 'total' in c:
        TOTAL = c['total']
        print('pendientes:', TOTAL, flush=True)
    if 'completadas' in c:
        print('  [{:>3}/{}] {}'.format(c['completadas'], TOTAL, c.get('actual', '')),
              flush=True)

def msj(m):
    print('   ', m, flush=True)

params = {
    'modo': 'local', 'ruta_local': CARPETA_FOTOS, 'salida': SALIDA,
    'pines': RUTA_PINES, 'drive_pics': '', 'kml': RUTA_KML, 'idx': RUTA_IDX,
    'n_fotos': N_FOTOS, 'radio': RADIO, 'cap': CAP, 'pausa': 1,
    'ia_proveedor': 'colab', 'ia_modelo': '',      # GPU local; si falla, baja por la cadena
    'trabajo': TRABAJO, '_mock_gps': None, '_detener': lambda: False,
}

t0 = time.time()
resumen = proceso.ejecutar_corrida(params, progreso, msj)
print('=' * 60)
print('TERMINADO en {:.0f} s'.format(time.time() - t0))
for k in ('leidas', 'copiadas', 'excedidas', 'idx'):
    print(' ', k, '=', resumen.get(k))
print('  pines:', ', '.join(resumen.get('pines') or []) or '—')
print('  grupos:', ', '.join(resumen.get('grupos') or []) or '—')

In [ ]:
#@title 7) Ver lo que se cargó (últimas filas con FechaCarga)
from openpyxl import load_workbook

wb = load_workbook(RUTA_IDX, read_only=True)
ws = wb['idxfotos'] if 'idxfotos' in wb.sheetnames else wb[wb.sheetnames[0]]
enc = [str(ws.cell(1, c).value or '').strip() for c in range(1, ws.max_column + 1)]
col_pin = (enc.index('PIN') + 1) if 'PIN' in enc else 9
col_ver = (enc.index('Verificado') + 1) if 'Verificado' in enc else 8
col_f = (enc.index('FechaCarga') + 1) if 'FechaCarga' in enc else None
filas = [r for r in range(2, ws.max_row + 1) if str(ws.cell(r, 1).value or '').strip()]
print('hoja:', ws.title, '| total filas:', len(filas))
print('columnas:', enc)
print('últimas 5:')
for r in filas[-5:]:
    print('  {:<20} PIN={:<24} Verificado={:<4} FechaCarga={}'.format(
        str(ws.cell(r, 1).value)[:20], str(ws.cell(r, col_pin).value),
        str(ws.cell(r, col_ver).value or '') or '-',
        ws.cell(r, col_f).value if col_f else 'SIN COLUMNA'))
wb.close()

## Si algo falla

| Error | Solución |
|---|---|
| `Sin GPU` / `CUDA out of memory` | Entorno ▸ Cambiar tipo de entorno ▸ GPU. Si sigue sin caber, en la celda 4 cambia el modelo a `Qwen/Qwen2.5-VL-3B-Instruct` |
| `No encuentro proceso.py` | Copia la carpeta `SuitCVLOGCPy` a Drive y corrige `PROYECTO` en la celda 3 |
| `RUTAS QUE NO EXISTEN` | Corrige las rutas en la celda 2 (deben ser las de tu Drive) |
| **Fotos repetidas** | No debería pasar: el mismo criterio de exclusión de la app local (`idxfotos.xlsx` + salida + `trabajo_colab/reads`). Si pasa, revisa que `RUTA_IDX` apunte al mismo xlsx en todas las celdas |
| `pillow_heif` / HEIC | Ya lo instala la celda 1; si falla, reinicia el entorno y empieza de nuevo |
| El modelo tarda en la 1ª foto | La primera generación incluye la carga a GPU (~20-60 s); las siguientes van ~1-3 s |
| Sesión de Colab expiró | Se repite la celda 1 → 2 → 3 → 4 → 6 (todo el estado está en Drive) |

**Nada se borra nunca**: los temporales quedan en `trabajo_colab/` y la salida es aditiva.
